# Titanic: обучение модели выживания пассажиров

Задача бинарной классификации: по данным пассажира предсказать, выжил ли он (`survived`).
Сохраняем **весь пайплайн** (предобработка + модель), чтобы сервис принимал сырые данные.

In [ ]:
import sys
sys.path.append('../src')

import joblib
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay, classification_report
from sklearn.model_selection import cross_val_score, train_test_split

import train
from model_utils import FEATURES, make_inference

## 1. Данные

In [ ]:
df = pd.read_csv('../data/titanic.csv')
print(df.shape)
df.head()

In [ ]:
df[FEATURES + ['survived']].info()
df[FEATURES].isna().sum()

Пропуски есть в `age` (~20%) и `embarked` (2 строки). Их заполнят импьютеры внутри пайплайна.

## 2. Разведочный анализ

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.barplot(data=df, x='sex', y='survived', ax=axes[0])
sns.barplot(data=df, x='pclass', y='survived', ax=axes[1])
sns.histplot(data=df, x='age', hue='survived', multiple='stack', bins=30, ax=axes[2])
axes[0].set_title('Доля выживших по полу')
axes[1].set_title('Доля выживших по классу')
axes[2].set_title('Возраст и выживание')
plt.tight_layout()

Пол и класс билета сильнее всего влияют на выживание («женщины и дети первыми»).

## 3. Сравнение моделей

Предобработка одинаковая: медиана + стандартизация для числовых признаков, мода + one-hot для категориальных.

In [ ]:
x, y = train.load_data('../data/titanic.csv')
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, stratify=y, random_state=train.RANDOM_STATE)

candidates = {
    'LogisticRegression': LogisticRegression(max_iter=1000),
    'RandomForest': RandomForestClassifier(n_estimators=300, max_depth=6, random_state=42),
    'GradientBoosting': GradientBoostingClassifier(n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42),
}
for name, clf in candidates.items():
    pipe = train.build_pipeline().set_params(classifier=clf)
    scores = cross_val_score(pipe, x_train, y_train, cv=5, scoring='accuracy')
    print(f'{name:20s} CV accuracy = {scores.mean():.3f} ± {scores.std():.3f}')

## 4. Итоговая модель: градиентный бустинг

In [ ]:
pipeline = train.build_pipeline().fit(x_train, y_train)
print(classification_report(y_test, pipeline.predict(x_test), digits=3))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ConfusionMatrixDisplay.from_estimator(pipeline, x_test, y_test, ax=axes[0])
RocCurveDisplay.from_estimator(pipeline, x_test, y_test, ax=axes[1])
plt.tight_layout()

## 5. Сохранение пайплайна

Финальную модель обучаем на всех данных и сохраняем целиком в `models/pipeline.pkl`.

In [ ]:
pipeline = train.build_pipeline().fit(x, y)
joblib.dump(pipeline, '../models/pipeline.pkl')

loaded = joblib.load('../models/pipeline.pkl')
print(make_inference(loaded, {'pclass': 1, 'sex': 'female', 'age': 29, 'sibsp': 0, 'parch': 0, 'fare': 211.34, 'embarked': 'S'}))
print(make_inference(loaded, {'pclass': 3, 'sex': 'male', 'age': None, 'sibsp': 0, 'parch': 0, 'fare': None, 'embarked': None}))